#반환 정보	역할	예시

job_id	청크와 그래프가 공유하는 직업 식별자	job:5

chunk_id	검색된 원문 조각의 식별자	career:5:workList:0

text	답변 작성·검증에 사용하는 실제 본문	일기예보관의 업무 설명

source_file, source_locator	원천 데이터에서 근거를 찾는 위치	cache/job_5.json, workList
[0].work

graph_paths	어떤 관계를 따라 찾았는지	학과 ← 직업 → 능력


관계별 출처	각 그래프 연결의 원천	departList[0] 등

셀 1 — PDF 제외 및 API 원천 검증

In [6]:
import csv
import hashlib
import json
import os
import re
import time
from collections import defaultdict

DATA = PROJECT_ROOT / "data" / "expanded_20261108"

all_chunks = json.loads(
    (DATA / "retrieval_chunks.json").read_text(encoding="utf-8")
)

# 이후 처리 대상에는 PDF가 들어가지 않습니다.
chunks = [
    dict(row) for row in all_chunks
    if row["source_type"] == "career_api"
]

if len({r["chunk_id"] for r in chunks}) != len(chunks):
    raise ValueError("중복 chunk_id가 있습니다.")

raw_cache = {}

def read_raw(source_file):
    if source_file not in raw_cache:
        path = (DATA / source_file).resolve()
        raw_cache[source_file] = json.loads(
            path.read_text(encoding="utf-8")
        )
    return raw_cache[source_file]

def resolve_locator(raw, locator):
    value = raw
    for part in locator.split("."):
        match = re.fullmatch(r"([^\[]+)(?:\[(\d+)\])?", part)
        if match is None:
            raise ValueError(f"지원하지 않는 출처 위치: {locator}")
        value = value[match.group(1)]
        if match.group(2) is not None:
            value = value[int(match.group(2))]
    return value

def normalized(text):
    return " ".join(str(text).split())

for row in chunks:
    raw_value = resolve_locator(
        read_raw(row["source_file"]), row["source_locator"]
    )
    if normalized(raw_value) != normalized(row["text"]):
        raise ValueError(f"원천 본문 불일치: {row['chunk_id']}")

print("사용할 API 청크:", len(chunks))       # 현재 파일: 6,798
print("직업:", len({r["job_code"] for r in chunks}))  # 552
print("API 원천 대조 완료")

사용할 API 청크: 6798
직업: 552
API 원천 대조 완료


셀 2 — 100개 표본 선택과 임베딩 입력 구성
major:111 관련 직업을 포함합니다. 현재 데이터에서는 관련 직업 5개에서 2개씩, 다른 직업에서 90개를 뽑아 100개 청크·95개 문서가 됩니다.

In [7]:
by_job = defaultdict(list)
for row in sorted(chunks, key=lambda r: r["chunk_id"]):
    by_job[row["job_code"]].append(row)

with (DATA / "graph_edges.csv").open(
    encoding="utf-8-sig", newline=""
) as file:
    edge_rows = list(csv.DictReader(file))

focus_codes = sorted({
    edge["start_id"].split(":", 1)[1]
    for edge in edge_rows
    if edge["relation"] == "관련학과"
    and edge["end_id"] == "major:111"
})

sample = [
    dict(row)
    for code in focus_codes
    for row in by_job[code][:2]
]

others = [
    rows[0] for code, rows in by_job.items()
    if code not in focus_codes
]
others.sort(
    key=lambda row: hashlib.sha256(
        row["chunk_id"].encode("utf-8")
    ).hexdigest()
)
sample += [dict(row) for row in others[:100 - len(sample)]]

assert len(sample) == len({r["chunk_id"] for r in sample}) == 100

for row in sample:
    row["job_id"] = f"job:{row['job_code']}"
    row["embedding_text"] = (
        f"직업: {row['job_name']}\n"
        f"항목: {row['source_locator']}\n"
        f"내용: {row['text']}"
    )
    row["input_hash"] = hashlib.sha256(
        row["embedding_text"].encode("utf-8")
    ).hexdigest()

sample_ids = [r["chunk_id"] for r in sample]
sample_by_id = {r["chunk_id"]: r for r in sample}

print("청크:", len(sample))
print("문서:", len({r["document_id"] for r in sample}))
print("표본 직업:", focus_codes)

청크: 100
문서: 95
표본 직업: ['1541', '31', '5', '6', '61']


#셀 3 — 충돌 검사 후 Document·Chunk 적재
기존 PDF가 이미 Aura에 있다면 여기서 삭제하지는 않습니다. 아래 검색은 API만 사용하도록 제한합니다.

In [8]:
job_ids = sorted({r["job_id"] for r in sample})

found = {
    r["id"] for r in run_cypher(
        "MATCH (j:Job) WHERE j.id IN $ids RETURN j.id AS id",
        ids=job_ids,
    )
}
if set(job_ids) - found:
    raise ValueError(f"Aura에 없는 직업: {set(job_ids) - found}")

existing = run_cypher("""
MATCH (c:Chunk)
WHERE c.chunk_id IN $ids
OPTIONAL MATCH (d:Document)-[:HAS_CHUNK]->(c)
OPTIONAL MATCH (c)-[:DESCRIBES]->(j:Job)
RETURN c.chunk_id AS id, c.text AS text,
       collect(DISTINCT d.document_id) AS documents,
       collect(DISTINCT j.id) AS jobs
""", ids=sample_ids)

for old in existing:
    expected = sample_by_id[old["id"]]
    if old["text"] != expected["text"]:
        raise ValueError(f"기존 본문 충돌: {old['id']}")
    if old["documents"] and old["documents"] != [expected["document_id"]]:
        raise ValueError(f"기존 문서 연결 충돌: {old['id']}")
    if old["jobs"] and old["jobs"] != [expected["job_id"]]:
        raise ValueError(f"기존 직업 연결 충돌: {old['id']}")

run_cypher("""
CREATE CONSTRAINT career_document_id IF NOT EXISTS
FOR (d:Document) REQUIRE d.document_id IS UNIQUE
""")
run_cypher("""
CREATE CONSTRAINT career_chunk_id IF NOT EXISTS
FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE
""")

query = """
UNWIND $rows AS row
MATCH (j:Job {id: row.job_id})
MERGE (d:Document {document_id: row.document_id})
SET d.source_type = 'career_api',
    d.source_file = row.source_file
MERGE (c:Chunk {chunk_id: row.chunk_id})
SET c.source_type = 'career_api',
    c.text = row.text,
    c.job_code = row.job_code,
    c.job_name = row.job_name,
    c.embedding_text = row.embedding_text,
    c.source_file = row.source_file,
    c.source_locator = row.source_locator,
    c.source_url_without_key = row.source_url_without_key,
    c.source_updated = row.source_updated
MERGE (d)-[:HAS_CHUNK]->(c)
MERGE (c)-[:DESCRIBES]->(j)
RETURN count(c) AS processed
"""

for start in range(0, len(sample), 25):
    batch = sample[start:start + 25]
    count = run_cypher(query, rows=batch)[0]["processed"]
    assert count == len(batch)

print("API 청크 100개와 문서·직업 연결 적재 완료")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `DESCRIBES` does not exist in database `2a55805a`. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=22, offset=109>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 109, 'line': 5, 'column': 22}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (c:Chunk)\nWHERE c.chunk_id IN $ids\nOPTIONAL MATCH (d:Document)-[:HAS_CHUNK]->(c)\nOPTIONAL MATCH (c)-[:DESCRIBES]->(j:Job)\nRETURN c.chunk_id AS id, c.text AS text,\n       collect(DISTINCT d.document_id) AS documents,\n       collect(DISTINCT j.id) AS jobs\n'
Received notification f

API 청크 100개와 문서·직업 연결 적재 완료


In [14]:
#| 173 | 조리사 및 주방장 | 외식조리과 — 학과 ID 377 | `departList[0]`, `[5]` |
#| 173 | 조리사 및 주방장 | 식품영양학과 — 학과 ID 318 | `departList[1]`, `[3]` |
#| 174 | 가축사육자 | 축산학과 — 학과 ID 557 | `departList[0]`, `[5]` |
#| 176 | 선장 및 항해사 | 항해학과 — 학과 ID 629 | `departList[3]`, `[6]` |
#| 177 | 푸드스타일리스트 | 외식조리과 — 학과 ID 377 | `departList[0]`, `[5]` |
#| 226 | 손해사정사 | 금융보험학과 — 학과 ID 83 | `departList[0]`, `[4]` |
#| 226 | 손해사정사 | 통계학과 — 학과 ID 591 | `departList[1]`, `[9]` |
#| 344 | 패션디자이너 | 시각디자인과 — 학과 ID 308 | `departList[2]`, `[8]` |
#| 1146 | 전기 부품 및 제품 생산직 | 전기산업기사 | `certiList[0]`, `[3]` |

셀 4 — 그래프 관계의 정확한 출처 보강
예를 들어 depart_name이라는 필드명만 있던 관계에 departList[0]과 그 원본 항목을 기록합니다. 원본 배열에서 이름이 유일하게 일치하지 않으면 중단합니다.

#9개의 중복 확인.
삭제할 문제는 아니고, 동일 항목의 반복을 처리하도록 검증 코드 수정

In [22]:
import json

RELATION_FIELDS = {
    "RELATED_MAJOR": ("departList", "depart_name"),
    "RELATED_CERTIFICATE": ("certiList", "certi"),
    "HAS_CORE_ABILITY": ("abilityList", "ability_name"),
}

edges = run_cypher("""
MATCH (j:Job)-[r]->(target)
WHERE type(r) IN $types
RETURN j.id AS job_id,
       target.id AS target_id,
       target.name AS target_name,
       type(r) AS relation,
       r.source_file AS source_file
""", types=list(RELATION_FIELDS))

evidence_rows = []
duplicate_count = 0

for edge in edges:
    source_file = edge["source_file"]
    if not source_file:
        raise ValueError(f"관계 출처 파일 누락: {edge}")

    raw = read_raw(source_file)
    list_name, name_field = RELATION_FIELDS[edge["relation"]]

    matches = [
        (i, item)
        for i, item in enumerate(raw.get(list_name, []))
        if normalized(item.get(name_field, ""))
        == normalized(edge["target_name"])
    ]

    # 학과는 이름뿐 아니라 원천 ID도 검증
    if edge["relation"] == "RELATED_MAJOR":
        matches = [
            (i, item)
            for i, item in matches
            if f"major:{item.get('depart_id')}" == edge["target_id"]
        ]

    # 동일 항목이 여러 번 등장하는 것은 허용
    if not matches:
        raise ValueError(f"관계에 대응하는 원천 항목이 없음: {edge}")

    index, item = matches[0]

    if len(matches) > 1:
        duplicate_count += 1

    evidence_rows.append({
        **edge,

        # 기존 검색 코드에서 사용할 대표 출처
        "source_locator": f"{list_name}[{index}]",
        "source_record_json": json.dumps(
            item, ensure_ascii=False
        ),

        # 같은 관계를 뒷받침하는 원천 위치와 항목 전체
        "source_locators": [
            f"{list_name}[{i}]" for i, _ in matches
        ],
        "source_records_json": json.dumps(
            [
                {
                    "source_locator": f"{list_name}[{i}]",
                    "record": record,
                }
                for i, record in matches
            ],
            ensure_ascii=False,
        ),
    })

print("원천 확인 완료:", len(evidence_rows), "개 관계")
print("원천 항목이 반복된 관계:", duplicate_count, "개")

# 위 검증이 모두 끝난 뒤 Aura에 출처 정보를 기록
processed_total = 0

for relation in RELATION_FIELDS:
    rows = [
        row for row in evidence_rows
        if row["relation"] == relation
    ]

    for start in range(0, len(rows), 100):
        batch = rows[start:start + 100]

        result = run_cypher(f"""
        UNWIND $rows AS row
        MATCH (j:Job {{id: row.job_id}})
              -[r:{relation}]->(target {{id: row.target_id}})
        SET r.source_locator = row.source_locator,
            r.source_record_json = row.source_record_json,
            r.source_locators = row.source_locators,
            r.source_records_json = row.source_records_json
        RETURN count(r) AS processed
        """, rows=batch)

        processed = result[0]["processed"]
        if processed != len(batch):
            raise RuntimeError(
                f"관계 갱신 건수 불일치: "
                f"요청 {len(batch)} / 처리 {processed}"
            )

        processed_total += processed

print("정확한 원천 위치를 보강한 관계:", processed_total)

원천 확인 완료: 4061 개 관계
원천 항목이 반복된 관계: 9 개
정확한 원천 위치를 보강한 관계: 4061


#셀 5 — 임베딩 및 API 전용 인덱스
PDF와 기존 벡터를 섞지 않도록 API 전용 라벨·속성·인덱스를 사용합니다. 모델은 text-embedding-3-small, 768차원입니다. 기존 벡터가 있으면 입력 해시를 검사해 재사용합니다.


In [16]:
from langchain_openai import OpenAIEmbeddings

MODEL = "text-embedding-3-small"
DIM = 768
INDEX = "career_api_chunk_768"

embedder = OpenAIEmbeddings(model=MODEL, dimensions=DIM)

state = {
    r["id"]: r
    for r in run_cypher("""
    MATCH (c:Chunk) WHERE c.chunk_id IN $ids
    RETURN c.chunk_id AS id,
           c.api_embedding IS NOT NULL AS ready,
           c.api_embedding_hash AS input_hash,
           c.api_embedding_model AS model,
           c.api_embedding_dim AS dimension
    """, ids=sample_ids)
}

pending = []
for row in sample:
    old = state[row["chunk_id"]]
    if old["ready"]:
        if (
            old["input_hash"] != row["input_hash"]
            or old["model"] != MODEL
            or old["dimension"] != DIM
        ):
            raise ValueError(f"既存 벡터 설정 충돌: {row['chunk_id']}")
    else:
        pending.append(row)

for start in range(0, len(pending), 20):
    batch = pending[start:start + 20]
    vectors = embedder.embed_documents(
        [r["embedding_text"] for r in batch]
    )
    assert len(vectors) == len(batch)
    assert all(len(v) == DIM for v in vectors)

    payload = [
        {
            "id": row["chunk_id"],
            "vector": vector,
            "input_hash": row["input_hash"],
        }
        for row, vector in zip(batch, vectors)
    ]

    result = run_cypher("""
    UNWIND $rows AS row
    MATCH (c:Chunk {chunk_id: row.id})
    SET c:CareerApiChunk,
        c.api_embedding = row.vector,
        c.api_embedding_hash = row.input_hash,
        c.api_embedding_model = $model,
        c.api_embedding_dim = $dimension
    RETURN count(c) AS saved
    """, rows=payload, model=MODEL, dimension=DIM)
    assert result[0]["saved"] == len(batch)

# 재실행 시에도 검색용 라벨 보장
run_cypher("""
MATCH (c:Chunk)
WHERE c.chunk_id IN $ids AND c.source_type = 'career_api'
SET c:CareerApiChunk
""", ids=sample_ids)

run_cypher(f"""
CREATE VECTOR INDEX {INDEX} IF NOT EXISTS
FOR (c:CareerApiChunk) ON (c.api_embedding)
OPTIONS {{indexConfig: {{
    `vector.dimensions`: 768,
    `vector.similarity_function`: 'cosine'
}}}}
""")

for _ in range(20):
    indexes = run_cypher("""
    SHOW VECTOR INDEXES
    YIELD name, state, labelsOrTypes, properties, options
    RETURN name, state, labelsOrTypes, properties, options
    """)
    index = next((r for r in indexes if r["name"] == INDEX), None)
    if index is None:
        raise RuntimeError("지정한 이름의 인덱스가 생성되지 않았습니다.")
    config = index["options"]["indexConfig"]
    if (
        index["labelsOrTypes"] != ["CareerApiChunk"]
        or index["properties"] != ["api_embedding"]
        or config.get("vector.dimensions") != DIM
        or config.get("vector.similarity_function", "").lower() != "cosine"
    ):
        raise ValueError("기존 인덱스 설정이 현재 코드와 다릅니다.")
    if index["state"] == "ONLINE":
        break
    if index["state"] == "FAILED":
        raise RuntimeError("벡터 인덱스 생성 실패")
    time.sleep(2)
else:
    raise TimeoutError("인덱스가 아직 준비 중입니다. 이 셀을 다시 확인하세요.")

print("임베딩·API 전용 인덱스 준비 완료")

임베딩·API 전용 인덱스 준비 완료


#셀 6 — 공통 근거 반환 함수

chunk_evidence는 원문 청크, graph_paths는 관계 및 관계의 원천입니다. 그래프만으로 찾은 직업에 아직 시험 청크가 없다면 chunk_evidence가 빈 목록일 수 있습니다. 그것을 검색 실패나 “원문 없음”으로 단정하지 않습니다.

In [17]:
def build_job_evidence(job_id, chunk_ids=None):
    chunk_query = """
    MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)
          -[:DESCRIBES]->(j:Job {id: $job_id})
    WHERE c.source_type = 'career_api'
      AND ($chunk_ids IS NULL OR c.chunk_id IN $chunk_ids)
    RETURN c.chunk_id AS chunk_id,
           d.document_id AS document_id,
           c.text AS text,
           c.source_file AS source_file,
           c.source_locator AS source_locator,
           c.source_url_without_key AS source_url
    ORDER BY c.chunk_id
    """
    chunk_evidence = run_cypher(
        chunk_query, job_id=job_id, chunk_ids=chunk_ids
    )

    paths = run_cypher("""
    MATCH (j:Job {id: $job_id})-[r]->(target)
    WHERE type(r) IN $types
    RETURN j.id AS from_id,
           type(r) AS relation,
           target.id AS to_id,
           target.name AS to_name,
           r.source_file AS source_file,
           r.source_locator AS source_locator,
           r.source_record_json AS source_record_json,
           r.source_url_without_key AS source_url
    ORDER BY relation, to_id
    """, job_id=job_id, types=list(RELATION_FIELDS))

    return {
        "job_id": job_id,
        "chunk_evidence": chunk_evidence,
        "graph_paths": paths,
        "has_loaded_text_evidence": bool(chunk_evidence),
    }

#셀 7 — 벡터 출발 검색

질문으로 청크를 찾고, 그 청크가 설명하는 Job의 그래프 관계를 함께 붙입니다.

In [23]:
def retrieve_vector(question, top_k=10):
    vector = embedder.embed_query(question)
    assert len(vector) == DIM

    hits = run_cypher("""
    CALL db.index.vector.queryNodes($index, $k, $vector)
    YIELD node AS c, score
    MATCH (c)-[:DESCRIBES]->(j:Job)
    WHERE c.source_type = 'career_api'
    RETURN c.chunk_id AS chunk_id,
           j.id AS job_id, j.name AS job_name, score
    ORDER BY score DESC
    """, index=INDEX, k=top_k, vector=vector)

    grouped = {}
    for hit in hits:
        item = grouped.setdefault(hit["job_id"], {
            "job_name": hit["job_name"],
            "score": hit["score"],
            "chunk_ids": [],
        })
        item["chunk_ids"].append(hit["chunk_id"])

    results = []
    for job_id, item in grouped.items():
        results.append({
            **build_job_evidence(job_id, item["chunk_ids"]),
            "job_name": item["job_name"],
            "vector_score": item["score"],
        })

    return {
        "query": question,
        "route": "vector_then_graph",
        "results": results,
    }

In [19]:
ENTITY_TYPES = {
    "major": ("Major", "RELATED_MAJOR"),
    "certificate": ("Certificate", "RELATED_CERTIFICATE"),
    "ability": ("Ability", "HAS_CORE_ABILITY"),
}

def retrieve_graph(entity_type, entity_name=None, entity_id=None, limit=10):
    if entity_type not in ENTITY_TYPES:
        raise ValueError(f"지원하지 않는 유형: {entity_type}")
    if not entity_name and not entity_id:
        raise ValueError("정확한 이름 또는 ID가 필요합니다.")

    label, relation = ENTITY_TYPES[entity_type]

    entities = run_cypher(f"""
    MATCH (e:{label})
    WHERE ($id IS NOT NULL AND e.id = $id)
       OR ($id IS NULL AND e.name = $name)
    RETURN e.id AS id, e.name AS name
    """, id=entity_id, name=entity_name)

    if len(entities) != 1:
        return {
            "route": "graph_then_text",
            "status": "entity_not_found" if not entities else "ambiguous_entity",
            "candidates": entities,
            "results": [],
        }

    entity = entities[0]
    jobs = run_cypher(f"""
    MATCH (j:Job)-[:{relation}]->(e:{label} {{id: $id}})
    RETURN DISTINCT j.id AS job_id, j.name AS job_name
    ORDER BY job_id
    LIMIT $limit
    """, id=entity["id"], limit=limit)

    results = [
        {
            **build_job_evidence(job["job_id"]),
            "job_name": job["job_name"],
        }
        for job in jobs
    ]

    return {
        "route": "graph_then_text",
        "status": "ok",
        "matched_entity": entity,
        "result_limit": limit,
        "results": results,
    }

#셀 8 — 그래프 출발 검색

여기서는 질문을 임의로 해석하지 않고 학과·자격·능력의 정확한 이름이나 ID를 받습니다. 질문에서 엔티티를 추출하는 역할은 이후 Supervisor 쪽과 연결

In [24]:
vector_result = retrieve_vector(
    "일기예보관은 어떤 일을 하나?",
    top_k=5,
)

graph_result = retrieve_graph(
    "major",
    entity_id="major:111",
    limit=10,
)

print("벡터 출발 결과")
print(json.dumps(vector_result, ensure_ascii=False, indent=2))

print("\n그래프 출발 결과")
print(json.dumps(graph_result, ensure_ascii=False, indent=2))

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes($index, $k, $vector)\n    YIELD node AS c, score\n    MATCH (c)-[:DESCRIBES]->(j:Job)\n    WHERE c.source_type = 'career_api'\n    RETURN c.chunk_id AS chunk_id,\n           j.id AS job_id, j.name AS job_name, score\n    ORDER BY score DESC\n    "


벡터 출발 결과
{
  "query": "일기예보관은 어떤 일을 하나?",
  "route": "vector_then_graph",
  "results": [
    {
      "job_id": "job:5",
      "chunk_evidence": [
        {
          "chunk_id": "career:5:aptitudeList:0",
          "document_id": "career:job:5",
          "text": "기상 관측 자료의 분석을 바탕으로 일기예보를 하기 위해서 논리적으로 사고하여 문제를 해결할 수 있는 수리·논리력이 필요합니다.",
          "source_file": "cache/job_5.json",
          "source_locator": "aptitudeList[0].aptitude",
          "source_url": "https://www.career.go.kr/cnet/front/openapi/job.json?seq=5"
        },
        {
          "chunk_id": "career:5:forecastList:0",
          "document_id": "career:job:5",
          "text": "향후 5년간 일기예보관의 관련 직업인 천문·기상학연구원의 일자리 규모는 다소 증가할 것으로 전망됩니다. 기초과학 분야의 연구 사업은 정부의 정책적 지원에 큰 영향을 받는데, 특히 정부가 국가경쟁력의 핵심 요소가 되는 과학기술 인력의 체계적 육성과 지원을 중요한 정책적 과제로 삼고 있고, 이에 수반되는 예산을 대폭 늘릴 예정이므로 향후 자연과학 관련 인력의 수요가 증가할 것으로 판단됩니다. 따라서 향후 천문 및 기상학연구원의 일자리도 다소 증가할 것으로 예측되고 있습니다. (자료: 워크넷 직업정보)",
          "source_file": "cache/job_5.json",
          "source_